# Airbus Ship Detection: Traditional Machine Learning vs. Deep Learning
### Course: UE24CS352A - Machine Learning Mini-Project
**Problem Statement:** Serial No. 35 - Airbus Ship Detection Benchmark
---
This notebook demonstrates the end-to-end pipeline: data loading, feature engineering (Hu Moments, Color Histogram, Haralick Textures), traditional ML model evaluation (10-fold CV), and threshold scanning.

In [ ]:
import os, sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.dataset import load_dataset_subset, generate_synthetic_samples, build_and_cache_features
from src.features import extract_features, extract_hu_moments, extract_color_histogram, extract_haralick_textures
from src.models_traditional import get_traditional_classifiers, train_and_evaluate_cv, scan_thresholds, evaluate_model_metrics
from src.train_traditional import run_feature_combination_experiments
from src.visualize import plot_feature_breakdown, plot_cross_validation_boxplot, plot_threshold_curve


## 1. Load Dataset & Inspect Sample Images

In [ ]:
images, labels, paths = load_dataset_subset('../data/sample_dataset', max_samples_per_class=100)
print(f'Loaded {len(images)} samples. (Ships: {np.sum(labels==1)}, No-Ships: {np.sum(labels==0)})')

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i in range(4):
    axes[i].imshow(images[i])
    axes[i].set_title('Ship' if labels[i] == 1 else 'No Ship')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 2. Feature Extraction Pipeline (Hu, HSV Color Histogram, Haralick GLCM Textures)

In [ ]:
sample_img = images[0]
hu = extract_hu_moments(sample_img)
his = extract_color_histogram(sample_img)
ha = extract_haralick_textures(sample_img)
all_feat = extract_features(sample_img, feature_set='all')

print(f'Hu Moments vector shape: {hu.shape}')
print(f'Color Histogram vector shape: {his.shape}')
print(f'Haralick Textures vector shape: {ha.shape}')
print(f'Total Combined Feature Vector dimension: {all_feat.shape}')

## 3. Feature Engineering Ablation Study (Reproducing Table 1)

In [ ]:
df_results = run_feature_combination_experiments(images, labels)
df_results

## 4. 10-Fold Stratified Cross-Validation

In [ ]:
X_all, _ = build_and_cache_features(images, labels, feature_set='all', cache_path=None)
cv_results = train_and_evaluate_cv(X_all, labels, n_splits=10)

for alg, res in cv_results.items():
    print(f'{alg:5s} -> Accuracy: {res["accuracy_mean"]*100:5.2f}% | Precision: {res["precision_mean"]*100:5.1f}% | Recall: {res["recall_mean"]*100:5.1f}%')

## 5. Visualizations

In [ ]:
plot_cross_validation_boxplot(cv_results, save_path='../models/cv_boxplot.png')
print('Saved CV Boxplot to models/cv_boxplot.png')